In [1]:
import sys
sys.path.append('..')
from parser.split import build_train_test_split

ratings, train, held_out = build_train_test_split()
print(f"Train: {len(train)}, Held-out: {len(held_out)}")

Train: 80896, Held-out: 19940


In [2]:
import pandas as pd

movies = pd.read_csv("../sample_data/inputs/ml-latest-small/movies.csv")
movies['genres'].head(10)

0    Adventure|Animation|Children|Comedy|Fantasy
1                     Adventure|Children|Fantasy
2                                 Comedy|Romance
3                           Comedy|Drama|Romance
4                                         Comedy
5                          Action|Crime|Thriller
6                                 Comedy|Romance
7                             Adventure|Children
8                                         Action
9                      Action|Adventure|Thriller
Name: genres, dtype: object

In [3]:
movies['genres_cleaned'] = movies['genres'].str.replace('|', ' ', regex=False) ## replacing | with spaces
movies[['title', 'genres', 'genres_cleaned']].head(10)

,title,genres,genres_cleaned
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy,Comedy
5,Heat (1995),Action|Crime|Thriller,Action Crime Thriller
6,Sabrina (1995),Comedy|Romance,Comedy Romance
7,Tom and Huck (1995),Adventure|Children,Adventure Children
8,Sudden Death (1995),Action,Action
9,GoldenEye (1995),Action|Adventure|Thriller,Action Adventure Thriller


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer ## TF-IDF vectorization 

tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(movies['genres_cleaned'])

print(genre_matrix.shape)

(9742, 24)


In [5]:
## Cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

content_similarity = cosine_similarity(genre_matrix)
print(content_similarity.shape)

(9742, 9742)


In [6]:
## Sanity check on a single row
## Finding movies similar to toy story movie


movie_idx = movies[movies['title'].str.contains('Toy Story', case=False)].index[0]
similar_scores = content_similarity[movie_idx]

similar_indices = similar_scores.argsort()[::-1][1:11]
movies.iloc[similar_indices][['title', 'genres']]

,title,genres
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
8927,The Good Dinosaur (2015),Adventure|Animation|Children|Comedy|Fantasy
6948,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy
8219,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy
6194,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy
9430,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
6486,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy


In [7]:
# Content-based score for one user

user_id = 1
liked_movies = train[(train['userId'] == user_id) & (train['rating'] >= 4)]['movieId'].tolist()
liked_indices = movies[movies['movieId'].isin(liked_movies)].index.tolist()

content_scores = content_similarity[liked_indices].mean(axis=0)
print(content_scores.shape)

(9742,)


In [8]:
from scoring.collaborative_filtering import build_cf_model

model, user_id_map, movie_id_map, movie_idx_to_id, user_item_matrix = build_cf_model(train)

c:\Users\Himanshu Jain\Downloads\recommendation-system\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Himanshu Jain\Downloads\recommendation-system\.venv\lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 20/20 [00:00<00:00, 50.89it/s]
